
# Station 类的气温统计与可视化（兼用兰州站示例数据）

本脚本演示面向对象编程在气象数据处理中的应用。它以兰州站（区站号
52889，纬 36.05°N、经 103.88°E、海拔约 1517 m）最近几日的逐日气温为
数据，展示三个要点：

1. 如何定义气象站 ``Station`` 类：用``__init__``保存站名、经纬度、海拔，
   用方法求气温的平均值、极值，并依据最高气温做等级判定；
2. 如何用 ``@property`` 把"由数据推算出的统计量"包装成只读属性；
3. 如何用面向对象绘图接口 ``fig, ax = plt.subplots()`` 把统计结果
   画成直观的折线与分布图。

脚本完全自包含：数据以"内联数组"写死在代码里，不读取任何外部文件，
直接运行即可看到结果。


## 一、中文显示配置
气象图里常出现中文标签（站名、℃ 等），需先配置中文字体并把负号
显示为 ASCII 减号，否则图上的负号会显示成方块。



In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False

## 二、定义气象站 Station 类
类的属性有站名(name)、区站号(station_id)、纬度(lat)、经度(lon)、
海拔(altitude)以及近几日逐日气温(temps)。方法用于完成统计与判定。



In [ ]:
class Station:
    """用一个"空气象观测站"类，负责保存台站信息并计算气温统计量。"""

    def __init__(self, name, station_id, lat, lon, altitude, temps=None):
        self.name = name
        self.station_id = station_id
        self.lat = lat
        self.lon = lon
        self.altitude = altitude
        self.temps = temps if temps is not None else []

    # -- @property：把方法伪装成只读属性，调用时不用加括号 ----
    @property
    def mean_temp(self):
        """日均气温：序列为空时返回空值，避免除零报错。"""
        if not self.temps:
            return float("nan")
        return sum(self.temps) / len(self.temps)

    @property
    def max_temp(self):
        """日最高气温（这几天里的极大值）。"""
        return max(self.temps)

    @property
    def min_temp(self):
        """日最低气温（这几天里的极小值）。"""
        return min(self.temps)

    # -- 普通方法 ---------------------------------------------
    def info(self):
        """返回台站基本信息字符串。"""
        return (f"{self.station_id} {self.name} "
                f"{self.lat}°N, {self.lon}°E, {self.altitude} m")

    @staticmethod
    def grade(temp):
        """依据日最高气温给出体感等级判定。

        阈值参考中央气象台高温预警与常见体感划分，仅用于教学演示。
        """
        if temp >= 35.0:
            return "高温"
        if temp >= 30.0:
            return "偏热"
        if temp >= 20.0:
            return "舒适"
        if temp >= 5.0:
            return "偏凉"
        return "寒冷"

    def level_counts(self):
        """统计这几天每个体感等级出现的天数，返回有序字典。"""
        from collections import Counter, OrderedDict
        counts = Counter(self.grade(t) for t in self.temps)
        order = ["高温", "偏热", "舒适", "偏凉", "寒冷"]
        return OrderedDict((k, counts.get(k, 0)) for k in order)

## 三、准备兰州站近几日气温数据（内联数组，无外部文件）



In [ ]:
dates = ["8-10", "8-11", "8-12", "8-13", "8-14", "8-15", "8-16", "8-17"]
temps = [33.2, 34.1, 32.5, 35.0, 36.8, 34.4, 31.9, 33.7]

# 创建兰州站对象
lanzhou = Station(
    name="兰州",
    station_id="52889",
    lat=36.05,
    lon=103.88,
    altitude=1517,
    temps=temps,
)

## 四、用类的方法做统计



In [ ]:
print(lanzhou.info())          # 台站基本信息
print(f"平均气温: {lanzhou.mean_temp:.1f} ℃")
print(f"最高气温: {lanzhou.max_temp} ℃")
print(f"最低气温: {lanzhou.min_temp} ℃")
print("等级天数:", dict(lanzhou.level_counts()))

## 五、面向对象绘图：fig, ax = plt.subplots()
用两联子图：左图看"日最高气温逐日变化"（折线），
右图看"这几天气温的分布"（直方图 + 统计标注）。



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# --- 左子图：逐日气温折线 ---
ax = axes[0]
ax.plot(dates, lanzhou.temps, marker="o", color="#c0392b", linewidth=2)
ax.axhline(lanzhou.mean_temp, color="#2c3e50", linestyle="--",
           linewidth=1, label=f"平均 {lanzhou.mean_temp:.1f}℃")
ax.axhline(lanzhou.max_temp, color="#27ae60", linestyle=":",
           linewidth=1, label=f"最高 {lanzhou.max_temp:.1f}℃")
ax.set_xlabel("日期")
ax.set_ylabel("日最高气温 / ℃")
ax.set_title(f"{lanzhou.name}站逐日日最高气温（{lanzhou.station_id}）")
ax.set_ylim(25, 40)
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=30)
ax.legend()

# --- 右子图：气温分布直方图 + 统计标注 ---
ax = axes[1]
n, bins, patches = ax.hist(lanzhou.temps, bins=5, color="#2980b9",
                           alpha=0.7, edgecolor="white")
# 依据等级为每根柱子着色，直观展示"偏热/舒适"占比
for rect, left in zip(patches, bins[:-1]):
    rect.set_color("#e74c3c" if Station.grade(left + (bins[1] - bins[0]) / 2) in
                   ("高温", "偏热") else "#f39c12")
ax.axvline(lanzhou.mean_temp, color="#2c3e50", linestyle="--", linewidth=1.5)
ax.set_xlabel("日最高气温 / ℃")
ax.set_ylabel("出现天数")
ax.set_title("日最高气温分布（等级着色）")
ax.grid(alpha=0.3)
ax.text(0.03, 0.95,
        f"平均 {lanzhou.mean_temp:.1f}℃\n最高 {lanzhou.max_temp:.1f}℃\n"
        f"最低 {lanzhou.min_temp:.1f}℃",
        transform=ax.transAxes, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.4", fc="#ecf0f1", alpha=0.9))

## 六、保存并显示



In [ ]:
fig.tight_layout()
fig.savefig("T506_station_temperature.png", dpi=150)
plt.show()

print("绘图完成，已保存为 T506_station_temperature.png")